In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [ ]:
cd "drive/MyDrive/WPI/3rd Year/A-B | Deep Learning | CS 541/DL Project/Code"

/content/drive/.shortcut-targets-by-id/1MmKwE-uT3TRPifUfMKOSAl93BK5Kr9oe/DL Project/Code


In [ ]:
cd "drive/MyDrive/DL Project/Code"

/content/drive/.shortcut-targets-by-id/1MmKwE-uT3TRPifUfMKOSAl93BK5Kr9oe/DL Project/Code


In [ ]:
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import numpy as np

# Define a custom dataset
class VariableLengthDataset(Dataset):
    def __init__(self, inputs, targets):
        """
        inputs: List of tensors of shape [N, 768] (variable N per sample)
        targets: List of tensors of shape [N] (variable N per sample)
        """
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

# Custom collate function for padding
def collate_fn(batch):
    """
    Pads sequences in a batch to the length of the longest sequence.
    """
    inputs, targets = zip(*batch)
    padded_inputs = pad_sequence(inputs, batch_first=True)  # Pads inputs to [batch_size, max_seq_len, 768]
    padded_targets = pad_sequence(targets, batch_first=True)  # Pads targets to [batch_size, max_seq_len]
    return padded_inputs, padded_targets

In [ ]:
import os
import pandas as pd

current_directory = os.getcwd()
data_dir = f'{current_directory}/data'
song_tensor_dir = f'{data_dir}/songs'
song_breaks_dir = f'{data_dir}/segment_breaks'


def progress_count(current, total):
    ending = '\n' if current == total else ''
    print(f'\r{current}/{total}', end=ending)


def expand_ones(arr, n):
    arr = np.array(arr)
    length = len(arr)
    expanded = np.zeros_like(arr).astype(float)

    # Iterate through the array
    for i in range(length):
        if arr[i] == 1:
            # Ensure boundaries are handled
            # start = max(0, i)
            # end = min(length, i + n + 1)
            # expanded[start:end] = 1

            expanded[i - 2] = min(1, expanded[i - 2] + 0.2)
            expanded[i - 1] = min(1, expanded[i - 1] + 0.35)
            expanded[i] = 1
            expanded[i + 1] = min(1, expanded[i + 1] + 1)
            expanded[i + 2] = min(1, expanded[i + 2] + 0.8)
            expanded[i + 3] = min(1, expanded[i + 3] + 0.25)

    return expanded


def get_dataset(segment_length_ms, left=0, right=80):
    sequences = []
    labels = []

    song_file_names = os.listdir(f'{song_tensor_dir}_{segment_length_ms}ms')
    tensor_files = filter(lambda x: x.endswith(".pt"), song_file_names)
    tensor_ids = sorted(map(lambda x: int(x[:-3]), tensor_files))[left:right]
    num_tensors = len(tensor_ids)

    for num, tensor_id in enumerate(tensor_ids):
        try:
            tensor = torch.load(f'{song_tensor_dir}_{segment_length_ms}ms/{tensor_id}.pt', weights_only=True)
            breaks = pd.read_csv(f'{song_breaks_dir}/{tensor_id}.csv')['break'][:tensor.size(0)].astype(int)
            breaks = expand_ones(breaks.values, 2)

            sequences.append(tensor)
            labels.append(torch.tensor(breaks, dtype=torch.float32))

            progress_count(num + 1, num_tensors)

        except FileNotFoundError:
          continue

    return VariableLengthDataset(sequences, labels)


In [ ]:
import torch
import torch.nn as nn

class SequenceTransformer(nn.Module):
    def __init__(self, input_dim=768, n_heads=8, num_layers=2, hidden_dim=512):
    # def __init__(self, input_dim=768, n_heads=4, num_layers=2, hidden_dim=252):
        super(SequenceTransformer, self).__init__()

        # Transformer encoder
        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=input_dim,
                nhead=n_heads,
                dim_feedforward=hidden_dim,
                activation='relu'
            ),
            num_layers=num_layers
        )

        # Linear head to project to scalar output per timestep
        self.output_layer = nn.Linear(input_dim, 1)

        # Sigmoid activation for output probabilities
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: [B, N, 768], expected input

        # Pass through transformer encoder
        encoded = self.transformer_encoder(x)  # [B, N, 768]

        # Remove batch dimension
        encoded = encoded.squeeze(1)  # [N, 768]

        # Linear projection to scalar output for each timestep
        logits = self.output_layer(encoded).squeeze(-1)  # [N]

        # Sigmoid activation for probabilities
        probs = self.sigmoid(logits)  # [N]

        return probs


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim

# Training function
def train_model(model, dataloader, criterion, optimizer, num_epochs=10):
    model.train()  # Set model to training mode
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for inputs, targets in dataloader:
            # Move data to device
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass
            outputs = model(inputs)

            # Compute loss
            loss = criterion(outputs, targets)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        model_directory = f'{current_directory}/pretrained/Transformer_Wave2Vec_Models'
        model_path = f'{model_directory}/model_{epoch + 1}_epoch_{epoch_loss / len(dataloader)}'
        torch.save(model.state_dict(), f'{model_path}')

        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss / len(dataloader):.4f}")


In [ ]:
# get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Create dataset and dataloader
dataset = get_dataset(20)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

# Initialize model, loss function, and optimizer
model = SequenceTransformer().to(device)
model_directory = f'{current_directory}/pretrained/Transformer_Wave2Vec_Models'
# model.load_state_dict(torch.load(f'{model_directory}/model_10_epoch_0.2973708310874842'))

criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Train the model
train_model(model, dataloader, criterion, optimizer, num_epochs=10)

cuda
0
1/801
2/802
3/803
4/804
5/805
6/806
7/807
8/808
9/809
10/80

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import confusion_matrix

model_directory = f'{current_directory}/pretrained/Transformer_Wave2Vec_Models'

model = SequenceTransformer()
# model.load_state_dict(torch.load(f'{model_directory}/model_6_epoch_0.31853928985232016', map_location=torch.device(device)))
# model.load_state_dict(torch.load(f'{model_directory}/model_10_epoch_0.3183817328032801', map_location=torch.device(device)))
model.load_state_dict(torch.load(f'{model_directory}/model_9_final', map_location=torch.device(device)))
# model.load_state_dict(torch.load(f'{model_directory}/model_8_epoch_0.3706449308132721', map_location=torch.device(device)))
model.to(device)
model.eval()

dataset = get_dataset(20, left=80, right=93)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)


cm = np.zeros((2, 2))

for inputs, targets in dataloader:
    # Move data to device
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)

    threshold = 0.45
    outputs = outputs.detach().cpu().numpy()
    outputs = (outputs >= threshold).astype(int).flatten()
    targets = targets.cpu().numpy()
    targets = (targets >= threshold).astype(int).flatten()

    cm += confusion_matrix(targets, outputs)

print(cm)

<ipython-input-43-cf6485455784>:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{model_directory}/model_9_final', map_location=torch.devi

13/13
[[49446.  6854.]
 [ 8328.  5228.]]


In [ ]:
model_directory = f'{current_directory}/pretrained/Transformer_Wave2Vec_Models'

model = SequenceTransformer()
model.load_state_dict(torch.load(f'{model_directory}/model_9_final', map_location=torch.device(device)))
model.to(device)
model.eval()

dataset = get_dataset(20, left=80, right=93)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

for inputs, targets in dataloader:
    # Move data to device
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)

    threshold = 0.45
    outputs = outputs.detach().cpu().numpy()
    outputs = (outputs >= threshold).astype(int).flatten()
    targets = targets.cpu().numpy()
    targets = (targets >= threshold).astype(int).flatten()

    for target in targets:
        print(target, end='')
    print("")
    for output in outputs:
        print(output, end='')
    print("\n\n")

<ipython-input-73-e0d21b1cea97>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{model_directory}/model_9_final', map_location=torch.devi

13/13
...




In [ ]:
def kms():

    model_directory = f'{current_directory}/pretrained/Transformer_Wave2Vec_Models'

    model = SequenceTransformer()
    model.load_state_dict(torch.load(f'{model_directory}/model_9_final', map_location=torch.device(device)))
    model.to(device)
    model.eval()

    song_file_names = os.listdir(f'{song_tensor_dir}_20ms')
    tensor_files = filter(lambda x: x.endswith(".pt"), song_file_names)
    tensor_ids = sorted(map(lambda x: int(x[:-3]), tensor_files))[80:93]
    num_tensors = len(tensor_ids)

    for num, tensor_id in enumerate(tensor_ids):
        print(tensor_id)
        try:
            tensor = torch.load(f'{song_tensor_dir}_20ms/{tensor_id}.pt', weights_only=True).to(device)
            outputs = model(tensor)

            threshold = 0.45
            outputs = outputs.detach().cpu().numpy()
            outputs = (outputs >= threshold).flatten().astype(int)

            for output in outputs:
                print(output, end='')
            print("\n\n")

        except FileNotFoundError:
            continue

kms()

<ipython-input-72-dae7888ae4cd>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{model_directory}/model_9_final', map_location=torch.devi

80
0000000000000000000000000000000000000000000000010000000000000000000000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000011100000000001000000000000000000000000000000000000000000000000001000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000110000000000000000000000001100000000000000000000010000000000000000000000000000000000000000000000000000000000000